In [15]:
import os
import uuid
import chromadb
import dotenv

from dotenv import  load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity
from langchain_core.documents import  Document
from langchain_community.document_loaders.text import TextLoader
from langchain_community.document_loaders.pdf import PyPDFLoader
from langchain_text_splitters import  RecursiveCharacterTextSplitter
from sentence_transformers import  SentenceTransformer
from langchain_openai import ChatOpenAI

In [2]:
def load_all_pdf():
    folder_path = r"C:\Users\pc\OneDrive\Music\Desktop\ML_Projects\Dataset"
    num_docs = 0
    all_docs = []

    for filname in os.listdir(folder_path):
        if filname.lower().endswith(".pdf"):
            # Complete file path
            pdf_path = os.path.join(folder_path, filname)
            
            loader = PyPDFLoader(pdf_path)
            doc = loader.load()

            all_docs.extend(doc)
            num_docs += 1
    print("total pdf:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [3]:
all_pdf_documents = load_all_pdf()

total pdf: 1
total pages: 15


In [4]:
# Split Documents

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap= chunk_overlap
    )
    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [5]:
chunks = split_docs(all_pdf_documents)
len(chunks)

93

In [6]:
# Embedding Manager

class EmbeddingManger:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        self.model_name = model_name
        print("Loading Model...", self,model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_sentence_embedding_dimension())

    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [7]:
embedding_manger = EmbeddingManger()

Loading Model... <__main__.EmbeddingManger object at 0x00000256ABAF4AD0> all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


embedding dimensions= 384


In [8]:
# Vector Store( CHORMADB )

class VectorStoreManger:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self.__initialize_store()

    def __initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)

        # Create a Client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # Create the Collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}

        )

        print("initalized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # Store => ids, embedding, documents, metadata
        ids = []
        all_metadata = []
        documentss_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_c{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documentss_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(ids=ids, metadatas=all_metadata, documents=documentss_content, embeddings=embeddings_list)

            print("totoal documents added in vector store=", len(documentss_content))
            print("docs in collection:", self.collection.count())

In [9]:
vector_store = VectorStoreManger()

initalized the vector store with collection: pdf
docs in collection: 186


In [10]:
texts = [doc.page_content for doc in chunks]
embeding = embedding_manger.generate_embeddings(texts)
vector_store.add_documents(chunks, embeding)

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

embeddings shape: (93, 384)
totoal documents added in vector store= 1
docs in collection: 187
totoal documents added in vector store= 2
docs in collection: 188
totoal documents added in vector store= 3
docs in collection: 189
totoal documents added in vector store= 4
docs in collection: 190
totoal documents added in vector store= 5
docs in collection: 191
totoal documents added in vector store= 6
docs in collection: 192
totoal documents added in vector store= 7
docs in collection: 193
totoal documents added in vector store= 8
docs in collection: 194
totoal documents added in vector store= 9
docs in collection: 195
totoal documents added in vector store= 10
docs in collection: 196
totoal documents added in vector store= 11
docs in collection: 197
totoal documents added in vector store= 12
docs in collection: 198
totoal documents added in vector store= 13
docs in collection: 199
totoal documents added in vector store= 14
docs in collection: 200
totoal documents added in vector store= 15


In [11]:
class RaGRetriver:
    def __init__(self, embedding_manger, vector_store):
        self.embedding_manger = embedding_manger
        self.vector_store = vector_store

    def retrive(self, query, tok_k=5, score_threshold=0.0):
        # Query => Embedding
        query_embeddings = self.embedding_manger.generate_embeddings([query])[0]

        # Sementic search
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()],
            n_results=tok_k
        )
        # cosine similarity
        retreved_dcos = []

        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]
            
            for i, (doc_id, metadata, documents, distances) in enumerate(zip(ids, metadatas, documents, distances)):
                similarity_score = 1 - distances

                if similarity_score >= score_threshold:
                    retreved_dcos.append({
                        "id": doc_id, 
                        "documents":documents, 
                        "metadata":metadata, 
                        "distance": documents, 
                        "similarty_score": similarity_score, 
                        "rank": i + 1
                    })
            print(f"retrived {len(retreved_dcos)} documents")
        else:
            print("no documents found")
        return retreved_dcos

In [12]:
rag_retriever = RaGRetriver(embedding_manger, vector_store)

In [13]:
rag_retriever.retrive("What is encoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrived 5 documents


[{'id': 'doc_c2da52458-437f-403d-a833-897a606a84e9',
  'documents': 'bottoms of the encoder and decoder stacks. The positional encodings have the same dimension dmodel\nas the embeddings, so that the two can be summed. There are many choices of positional encodings,\nlearned and fixed [9].\nIn this work, we use sine and cosine functions of different frequencies:\nP E(pos,2i) = sin(pos/100002i/dmodel)\nP E(pos,2i+1) = cos(pos/100002i/dmodel)\nwhere pos is the position and i is the dimension. That is, each dimension of the positional encoding',
  'metadata': {'subject': '',
   'total_pages': 15,
   'author': '',
   'page': 5,
   'page_label': '6',
   'content_length': 471,
   'trapped': '/False',
   'creationdate': '2024-04-10T21:11:43+00:00',
   'doc_index': 37,
   'keywords': '',
   'creator': 'LaTeX with hyperref',
   'producer': 'pdfTeX-1.40.25',
   'source': 'C:\\Users\\pc\\OneDrive\\Music\\Desktop\\ML_Projects\\Dataset\\1706.03762v7.pdf',
   'title': '',
   'ptex.fullbanner': 'This

In [28]:
# Define api 


load_dotenv()
api_key = os.getenv("OPENAI_API_KEY")

print("API Loaded")

API Loaded


In [29]:
# Define LLM
llm = ChatOpenAI(
    openai_api_key = api_key,
    model = "gpt-5.4",
    temperature=0.1,
    max_tokens=1024
)

In [30]:
# Generate our retrieval-augmented output

def generate_output(query, retriever, llm, top_k= 3):
    results = retriever.retrive(query, top_k)

    context = "\n".join([doc["documents"] for doc in results]) if results else ""

    if not context:
        print("we found no relevant context fot the given query")

    # Context + Query
    prompt = f""" use given context to generate the answer for the query 
                 context: {context}
                Query: {query}"""

    response = llm.invoke(prompt)   # expecting a string as prompt
    return response.content

In [31]:
answer = generate_output("what is encoder-decoder", rag_retriever, llm)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrived 3 documents


In [32]:
print(answer)

In the given context, the **encoder-decoder** setup refers to the Transformer architecture having:

- an **encoder** stack and
- a **decoder** stack.

Specifically, the **decoder**:
- is made of **6 identical layers**,
- includes an extra **third sub-layer** that performs **multi-head attention over the output of the encoder stack**,
- uses **residual connections** and **layer normalization**,
- and modifies self-attention so it **cannot attend to future positions**.

So, **encoder-decoder attention** is the part where the decoder attends to the **encoder’s output** to use information from the input sequence while generating the output sequence.
